# M2 · 03 — Data Cleaning

This notebook runs the cleaning pipeline in `src/medibridge/cleaning.py`, the same code that `make prepare` (`medibridge.pipeline`) uses to write `data/processed/`. For each step it shows **why** the step is needed, **what changed**, and **proof that the entity offsets are still correct**.

| Step | Rule | Why |
|---|---|---|
| 1 | Drop empty transcripts | Nothing to learn from |
| 2 | Validate raw offsets: `text[start:end] == entity` | Annotation errors would silently corrupt training |
| 3 | Unicode NFC, remove zero-width characters, unify quotes and `॥` | The same word can have several byte forms in Devanagari |
| 4 | Collapse repeated whitespace and strip, **remapping every offset** | Changing text shifts character offsets |
| 5 | Flatten nested or repeated spans: keep the longest span, store inner spans in `nested_entities` | BIO tagging needs non-overlapping spans, but no information is thrown away |
| 6 | Mask self-introduced person names with `[NAME]` | Privacy (see `06_privacy_considerations.md`) |
| 7 | Drop duplicate transcripts (case-insensitive) | Duplicates add no information and cause train/test leakage |
| 8 | Flag rows with no entities, audio under 1 s, residual PII | Kept, but visible for modelling decisions |
| 9 | Add features: tokens, entity count, script shares, CMI | Used in the statistics and split stratification |
| 10 | Export span JSONL and token-level BIO (CoNLL) | Ready for baseline NER in M3 |

In [1]:
# Requires the project package: `pip install -e ".[analysis]"` from the repository root.
import json
import re
from pathlib import Path

import pandas as pd
import seaborn as sns

from medibridge.bio import to_bio, unaligned_spans
from medibridge.cleaning import clean
from medibridge.config import PROCESSED_DIR, SUBSETS
from medibridge.io import load_raw
from medibridge.privacy import pii_scan
from medibridge.spans import parse_entities, resolve_overlaps
from medibridge.text import normalize_text

In [2]:
sns.set_theme(style="whitegrid", context="notebook", palette="deep")
FIG = Path("figures")
FIG.mkdir(exist_ok=True)
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 160)

## Run the pipeline on both subsets

In [3]:
raw = {s: load_raw(s) for s in SUBSETS}
cleaned, reports = {}, {}
for s, d in raw.items():
    cleaned[s], reports[s] = clean(d)
pd.DataFrame(reports)

,hi,en
input_rows,320,3619
dropped_empty_text,0,0
texts_changed_by_normalisation,69,120
nested_spans_moved_to_nested_entities,85,329
rows_with_names_masked,2,0
spans_validated,1760,10798
dropped_duplicate_text,4,292
duplicate_groups_with_conflicting_labels,0,54
flag_no_entities,10,10
flag_short_audio,0,8


Every one of the validated spans in the table matched its text **before and after** cleaning. `clean()` raises an error if any span breaks, so this table is the proof.

## Step 3–4: normalisation, with offsets remapped
Examples of transcripts that normalisation changed (whitespace and invisible characters made visible):

In [4]:
changed = []
for r in raw["hi"].itertuples():
    spans = parse_entities(r.medical_entities)
    new, new_spans = normalize_text(r.text, spans)
    if new != r.text:
        changed.append(
            {
                "before": repr(r.text[:90]),
                "after": repr(new[:90]),
                "precomposed nukta letters": len(re.findall("[क़-य़]", r.text)),
                "repeated spaces": len(re.findall("  +", r.text)),
                "zero-width chars": len(re.findall("[​﻿⁠]", r.text)),
                "length change": len(new) - len(r.text),
                "spans_still_valid": all(
                    new[e["start"] : e["end"]] == e["text"] for e in new_spans
                ),
            }
        )
changed = pd.DataFrame(changed)
print(changed[["precomposed nukta letters", "repeated spaces", "zero-width chars"]].sum().to_dict())
changed.head(6)

{'precomposed nukta letters': 109, 'repeated spaces': 0, 'zero-width chars': 0}


,before,after,precomposed nukta letters,repeated spaces,zero-width chars,length change,spans_still_valid
0,'बंद नहीं हुआ है । और इसके साथ बुखार या उल्टी कुछ हो रहा है? नहीं नहीं । थोड़ा सा','बंद नहीं हुआ है । और इसके साथ बुखार या उल्टी कुछ हो रहा है? नहीं नहीं । थोड़ा सा',1,0,0,1,True
1,"'Hello, Hi Doctor. Hi Rishabh, How are you feeling? सर दर्द हो रहा है आज सुबह से मुझे । कब '","'Hello, Hi Doctor. Hi Rishabh, How are you feeling? सर दर्द हो रहा है आज सुबह से मुझे । कब '",1,0,0,1,True
2,"'कौन सी, कौन सी, कौन सी कौनसी एंटीबायोटिक्स सर ? ज़ीफि टू हंड्रेड । अच्छा ये दिन में कितनी ब'","'कौन सी, कौन सी, कौन सी कौनसी एंटीबायोटिक्स सर ? ज़ीफि टू हंड्रेड । अच्छा ये दिन में कितनी '",2,0,0,2,True
3,"'वॉमिट टाइप फ़ील हो रहा है बट कुछ हुआ नहीं है । और कोई Do you smoke or drink alcohol? Yes, I'","'वॉमिट टाइप फ़ील हो रहा है बट कुछ हुआ नहीं है । और कोई Do you smoke or drink alcohol? Yes, '",1,0,0,1,True
4,"'पेट में दर्द हो रहा है तो तुम्हे मैं दवाई दे देता हूँ दवाई। तुम, एक तो अल्ट्रासाउंड करा के'","'पेट में दर्द हो रहा है तो तुम्हे मैं दवाई दे देता हूँ दवाई। तुम, एक तो अल्ट्रासाउंड करा के'",2,0,0,2,True
5,"'स्ट्रेस ज्यादा कर लो तो, पर अभी पिछले एक डेढ़ हफ्ते से थोड़ा ज्यादा फ्रीक्वेंटली होने लगा है'","'स्ट्रेस ज्यादा कर लो तो, पर अभी पिछले एक डेढ़ हफ्ते से थोड़ा ज्यादा फ्रीक्वेंटली होने लगा '",3,0,0,3,True


**What changed:** most edits are **precomposed nukta letters** (e.g. `ज़` stored as the single code point U+095B). Unicode NFC rewrites these as `ज` + `़`, so both spellings of the same word become identical bytes. That is why some texts get *longer* by a character. The other edits are repeated spaces and invisible zero-width characters. In every changed transcript the entity spans are still valid.

## Step 5: nested and repeated annotations
The raw labels contain nested spans, e.g. `Augmentin 650 mg` (drugs) containing `Augmentin` (drugs), and spans of another type inside an `advices` span. We keep the longest span as the main label and store the inner ones in `nested_entities`, so a nested-NER model can still use them later.

In [5]:
nested_rows = []
for r in raw["hi"].itertuples():
    kept, nested = resolve_overlaps(parse_entities(r.medical_entities))
    for n in nested:
        outer = next(
            k
            for k in kept
            if (k["start"] <= n["start"] and n["end"] <= k["end"])
            or (k["start"] < n["end"] and n["start"] < k["end"])
        )
        nested_rows.append(
            {
                "outer (kept)": outer["text"],
                "outer type": outer["type"],
                "inner (nested)": n["text"],
                "inner type": n["type"],
            }
        )
nested_df = pd.DataFrame(nested_rows)
print(
    "nested spans in hi:",
    len(nested_df),
    "| inner type differs from outer:",
    int((nested_df["outer type"] != nested_df["inner type"]).sum()),
)
nested_df.drop_duplicates().head(10)

nested spans in hi: 85 | inner type differs from outer: 14


,outer (kept),outer type,inner (nested),inner type
0,CBC lab test,diagnostics,lab test,diagnostics
1,viral fever,clinical_findings,fever,clinical_findings
3,medicines,drugs,medicine,drugs
4,Benadryl cough syrup,drugs,cough syrup,drugs
5,DO tablet,drugs,tablet,drugs
6,Voveran 5 mg injection,drugs,injection,drugs
7,"Oily food, जो भी है मत खाइए",advices,Oily food,misc_medical
8,body pain,clinical_findings,pain,clinical_findings
10,डायबिटीज की दवाई,drugs,डायबिटीज,clinical_findings
11,ग्लाइकोमेट जीपी,drugs,ग्लाइकोमेट,drugs


## Step 6: privacy masking
A PII scan looks for phone numbers, e-mails, Aadhaar-like 12-digit numbers and URLs, and for self-introductions such as *"मेरा नाम … है"* ("my name is …"). The name is replaced with `[NAME]` everywhere in that transcript, and entity offsets are shifted. Only the masked text is shown here.

In [6]:
for s in SUBSETS:
    hits = [pii_scan(t) for t in raw[s]["text"]]
    print(
        s,
        "| identifier patterns found in raw text:",
        sum(bool(h) for h in hits),
        "| transcripts with a masked name:",
        reports[s]["rows_with_names_masked"],
    )
masked = cleaned["hi"][cleaned["hi"]["text"].str.contains(r"\[NAME\]", regex=True)]
for t in masked["text"]:
    i = t.index("[NAME]")
    print("…", t[max(0, i - 40) : i + 60], "…")

hi | identifier patterns found in raw text: 0 | transcripts with a masked name: 2
en | identifier patterns found in raw text: 0 | transcripts with a masked name: 0
… ्स योर नेम? नमस्ते डॉक्टर साहब मेरा नाम [NAME] है। [NAME] जी कैसा आना हुआ आपका आज? मुझे ना पेट में द …
… लो, क्या नाम है जी आपका? अह मेरा नाम है [NAME]। ओके । ओके। [NAME] जी, क्या क्या हो गया आपको? कैसे आन …


## Step 7: duplicates
In `en`, many single-term narrations are the same word read by different speakers. For text NER they are identical rows, so one copy is kept. We also check whether duplicate copies were labelled differently.

In [7]:
key = raw["en"]["text"].str.lower()
dups = (
    raw["en"][key.duplicated(keep=False)]
    .assign(k=key)
    .groupby("k")
    .agg(copies=("text", "size"), example=("text", "first"))
)
dups.sort_values("copies", ascending=False).head(8)

,copies,example
k,,
cipla limited,7,Cipla Limited
sun pharmaceutical industries limited,7,Sun Pharmaceutical Industries Limited
torrent pharmaceuticals limited,6,Torrent Pharmaceuticals limited
lupin limited,5,Lupin Limited
ipca laboratories limited,5,Ipca Laboratories limited
sanofi india limited,5,Sanofi India Limited
abbott,5,Abbott
mankind pharma limited,5,Mankind Pharma limited


## Step 8–9: flags and engineered features

In [8]:
cols = [
    "text",
    "context",
    "n_tokens",
    "n_entities",
    "pct_devanagari",
    "pct_latin",
    "cmi",
    "flag_no_entities",
    "flag_short_audio",
]
cleaned["hi"][cols].sample(5, random_state=1)

,text,context,n_tokens,n_entities,pct_devanagari,pct_latin,cmi,flag_no_entities,flag_short_audio
165,हेपबेस्ट 25 एमजी विवरण हेपबेस्ट टैबलेट का इस्तेमाल ह्यूमन इम्यूनोडेफिशिएंसी वायरस (एचआईवी) इन्फेक्शन से पी...,narration_sentence,37,6,0.8649,0.0,0.0,False,False
161,"ओलवांस 40 एमजी विवरण ओलवांस टैब्लेट में ओल्मेसार्टन मेडोक्सोमिल, एंटी-हाइपरटेंसिव होता है। इसका इस्तेमाल ह...",narration_sentence,45,6,0.8444,0.0,0.0,False,False
271,"डैन एमआर 10 टैबलेट्स की स्ट्रिप विवरण डैन-एमआर टैबलेट में तीन दवाओं का मिश्रण होता है - डाइक्लोफेनेक, पैरा...",narration_sentence,42,7,0.8571,0.0,0.0,False,False
201,अम्ब्रिकन 5 एमजी विवरण अम्ब्रिकन टैबलेट को पल्मोनरी आर्टीरियल हाइपरटेंशन (पीएएच) के लिए निर्धारित किया जात...,narration_sentence,39,6,0.8718,0.0,0.0,False,False
186,मेटफिल वीजी 2एमजी 15 टैबलेट की स्ट्रिप विवरण मेटफिल वीजी टैब्लेट एक एंटी-डायबिटीज दवा है जिसका इस्तेमाल टा...,narration_sentence,32,4,0.8750,0.0,0.0,False,False


In [9]:
pd.DataFrame(
    {s: d[["flag_no_entities", "flag_short_audio", "flag_pii"]].sum() for s, d in cleaned.items()}
)

,hi,en
flag_no_entities,10,10
flag_short_audio,0,8
flag_pii,0,0


## Step 10: model-ready formats
**Span JSONL** (one transcript per line) keeps exact character offsets, for span-based models like GLiNER and for evaluation. **BIO CoNLL** is for token-classification baselines. Spans whose edge falls inside a token cannot be represented exactly in BIO; we count them:

In [10]:
for s, d in cleaned.items():
    n = sum(len(unaligned_spans(t, e)) for t, e in zip(d.text, d.entities))
    print(
        f"{s}: {n} of {int(d.n_entities.sum())} spans not token-aligned ({n / d.n_entities.sum():.2%})"
    )
r = cleaned["hi"][cleaned["hi"].context == "conversation"].iloc[0]
pd.DataFrame(to_bio(r.text, r.entities)[:25], columns=["token", "BIO tag"]).T

hi: 2 of 1733 spans not token-aligned (0.12%)
en: 11 of 10309 spans not token-aligned (0.11%)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
token,and,उसको,daily,सुबह,शाम,में,लगाना,।,And,medicines,दे,रहा,हूँ,मैं,थोड़ा,.,Augmentin,650,mg,.
BIO tag,O,O,B-advices,B-advices,I-advices,O,O,O,O,B-drugs,O,O,O,O,O,O,B-drugs,I-drugs,I-drugs,O


In [11]:
example = json.loads((PROCESSED_DIR / "hi_train.jsonl").read_text(encoding="utf-8").splitlines()[0])
{
    k: (v if k not in ("text",) else v[:120] + "…")
    for k, v in example.items()
    if k != "nested_entities"
}

{'id': '942d0c98d850b1d5f471c17130643e89',
 'subset': 'hi',
 'text': 'समझ गया । किसके कारण होगा ये? It could be stress related or migraine,अच्छा। Make sure you are hydrating yourself well और…',
 'entities': [{'start': 42,
   'end': 48,
   'text': 'stress',
   'type': 'clinical_findings'},
  {'start': 60, 'end': 68, 'text': 'migraine', 'type': 'clinical_findings'},
  {'start': 94, 'end': 103, 'text': 'hydrating', 'type': 'advices'},
  {'start': 137, 'end': 149, 'text': 'खाना टाइम पे', 'type': 'advices'},
  {'start': 162,
   'end': 187,
   'text': 'नींद अटलीस्ट 6 टू 8 हॉर्स',
   'type': 'advices'},
  {'start': 214, 'end': 227, 'text': 'avoid alcohol', 'type': 'advices'}],
 'session_id': '5482F557-6E87-40B1-BA39-B0C93872533B',
 'speaker': 'unknown',
 'context': 'conversation',
 'type_concept': 'misc_medical',
 'duration': 20.4,
 'md5_text': '1ae62262f1e93e81ff73e92bd9305f21',
 'n_tokens': 54,
 'n_entities': 6,
 'pct_devanagari': 0.4815,
 'pct_latin': 0.3519,
 'cmi': 42.22,
 'pii': {},
 'f

## Before → after summary

In [12]:
summary = pd.DataFrame(
    {
        s: {
            "rows before": reports[s]["input_rows"],
            "rows after": reports[s]["output_rows"],
            "entity spans before (incl. nested & repeats)": reports[s]["spans_validated"],
            "main entity spans after": reports[s]["output_entities"],
            "nested spans kept separately": reports[s]["nested_spans_moved_to_nested_entities"],
            "duplicates removed": reports[s]["dropped_duplicate_text"],
            "texts normalised": reports[s]["texts_changed_by_normalisation"],
            "names masked": reports[s]["rows_with_names_masked"],
        }
        for s in SUBSETS
    }
)
summary

,hi,en
rows before,320,3619
rows after,316,3327
entity spans before (incl. nested & repeats),1760,10798
main entity spans after,1733,10309
nested spans kept separately,85,329
duplicates removed,4,292
texts normalised,69,120
names masked,2,0
